In [1]:
%cd ..

/Users/emir.demic/Desktop/PhD/Research/Irena 2025/moral-reasoning-2026a


In [2]:
import os
from pprint import pprint

import pandas as pd
import numpy as np
import pingouin as pn
import statsmodels.api as sm

from src.descriptives import descriptive_stat, scale_stat
from src.util import value_labels
from src.models.regression import bootstrap_regression
import scipy.stats as stats
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [3]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [4]:
short_df = pd.read_csv("data/processed/01_short_form_data.csv")
long_df = pd.read_csv("data/processed/01_long_form_data.csv")

In [5]:
short_df

,Participant_ID,age,gender,education,residence,ses,MFQ2_C_1,MFQ2_C_2,MFQ2_C_3,MFQ2_C_4,...,MFQ_A,MFQ_P,CRT1_correct,CRT2_correct,CRT3_correct,CRT4_correct,CRT5_correct,CRT6_correct,CRT7_correct,CRT8_correct
0,727,38.0,1,4,3,3,5,4,5,5,...,4.000000,4.000000,0,0,1,1,0,0,0,1
1,88,30.0,1,4,4,2,3,2,3,2,...,3.166667,3.166667,0,0,0,1,0,0,0,0
2,544,51.0,1,6,1,4,5,5,3,5,...,4.500000,4.166667,0,0,0,1,0,0,0,1
3,115,50.0,1,3,1,1,5,5,5,5,...,2.666667,4.000000,1,0,0,1,0,0,0,0
4,396,40.0,1,4,4,3,5,1,5,5,...,3.666667,3.000000,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
756,627,24.0,2,5,4,4,4,4,5,3,...,2.166667,2.000000,1,1,1,1,1,1,0,1
757,641,42.0,1,4,4,3,5,5,5,5,...,3.833333,3.500000,0,0,0,1,0,0,0,0
758,653,30.0,1,5,1,4,4,4,4,4,...,2.000000,1.666667,0,0,0,1,1,0,0,0
759,705,23.0,1,3,1,4,5,5,5,5,...,1.500000,1.166667,0,0,1,1,1,1,0,1


# Deliberation and moral norms

In [6]:
correlation_results = {}
for i in ["MFQ_C", "MFQ_E", "MFQ_PRO", "MFQ_L", "MFQ_A", "MFQ_P"]:
    for j in ["AOT", "CM", "PIT", "PE", "CRT_Correct", "CRT_Heuristic"]:
        corr = pn.corr(short_df[i], short_df[j])
        correlation_results[(i, j)] = {
            "r": round(corr.at["pearson", "r"], 2),
            "CI95%": corr.at["pearson", "CI95%"],
            "p-val": round(corr.at["pearson", "p-val"], 3),
        }

In [7]:
correlation_results

{('MFQ_C', 'AOT'): {'r': np.float64(0.03),
  'CI95%': array([-0.04,  0.1 ]),
  'p-val': np.float64(0.451)},
 ('MFQ_C', 'CM'): {'r': np.float64(0.08),
  'CI95%': array([0.01, 0.15]),
  'p-val': np.float64(0.024)},
 ('MFQ_C', 'PIT'): {'r': np.float64(0.09),
  'CI95%': array([0.02, 0.16]),
  'p-val': np.float64(0.01)},
 ('MFQ_C', 'PE'): {'r': np.float64(0.02),
  'CI95%': array([-0.05,  0.1 ]),
  'p-val': np.float64(0.499)},
 ('MFQ_C', 'CRT_Correct'): {'r': np.float64(-0.08),
  'CI95%': array([-0.16, -0.01]),
  'p-val': np.float64(0.019)},
 ('MFQ_C', 'CRT_Heuristic'): {'r': np.float64(0.06),
  'CI95%': array([-0.01,  0.13]),
  'p-val': np.float64(0.09)},
 ('MFQ_E', 'AOT'): {'r': np.float64(-0.09),
  'CI95%': array([-0.16, -0.01]),
  'p-val': np.float64(0.019)},
 ('MFQ_E', 'CM'): {'r': np.float64(0.12),
  'CI95%': array([0.05, 0.19]),
  'p-val': np.float64(0.001)},
 ('MFQ_E', 'PIT'): {'r': np.float64(0.16),
  'CI95%': array([0.09, 0.23]),
  'p-val': np.float64(0.0)},
 ('MFQ_E', 'PE'): {'r':

In [8]:
regressions = {}
predictors = ["AOT", "CM", "PIT", "PE", "CRT_Correct", "CRT_Heuristic"]
for i in ["MFQ_C", "MFQ_E", "MFQ_PRO", "MFQ_L", "MFQ_A", "MFQ_P"]:
    full_X = short_df[predictors]
    full_X = sm.add_constant(full_X)
    full_model = sm.OLS(short_df[i], full_X, hasconst=True).fit()
    regressions[i] = {"full_model": full_model}
    regressions[i]["bootstrap"] = bootstrap_regression(full_model, full_X, short_df[i])

In [9]:
largest_coefficient_difference = 0
dep_var = None
input_var = None
for var, results in regressions.items():
    bootstrap_result = results["bootstrap"]
    for i, j in bootstrap_result["original_coef"].items():
        if i == "const":
            continue
        difference = abs(abs(j) - abs(bootstrap_result.at[i, "boot_mean"]))
        if difference > largest_coefficient_difference:
            largest_coefficient_difference = difference
            dep_var = var
            input_var = i
print(f"Largest coefficient difference was for {input_var} ({round(largest_coefficient_difference, 4)}) for the model {dep_var}.")

Largest coefficient difference was for PE (0.0007) for the model MFQ_L.


In [10]:
for i, j in regressions.items():
    print(i, j["full_model"].summary())
    print("\n\n")

MFQ_C                             OLS Regression Results                            
Dep. Variable:                  MFQ_C   R-squared:                       0.028
Model:                            OLS   Adj. R-squared:                  0.021
Method:                 Least Squares   F-statistic:                     3.681
Date:                Mon, 19 Jan 2026   Prob (F-statistic):            0.00130
Time:                        18:52:46   Log-Likelihood:                -790.70
No. Observations:                 761   AIC:                             1595.
Df Residuals:                     754   BIC:                             1628.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             3.3170      0.267     

In [11]:
all_coef_results = {}
all_model_results = pd.DataFrame(index=["R2", "Adj_R2", "F", "p_value", "df1", "df2"])
for i, j in regressions.items():
    model = j["full_model"]
    all_coef_results[i] = pd.DataFrame(
        data={
            "param": round(model.params, 2),
            "se": round(model.bse, 2),
            "p_value": round(model.pvalues, 4),
        }
    )
    all_model_results[i] = pd.Series(
        data={
            "R2": round(model.rsquared, 3),
            "Adj_R2": round(model.rsquared_adj, 3),
            "F": round(model.fvalue, 3),
            "p_value": round(model.f_pvalue, 4),
            "df1": model.df_model,
            "df2": model.df_resid,
        }
    )
all_coef_results = pd.concat(all_coef_results, axis=1)
all_coef_results.to_csv("results/coefficients/02_mfq2_regression.csv")
all_model_results.to_csv("results/coefficients/02_mfq2_model_results.csv")

In [12]:
# Exploring multicollinearity

ind_variables = [
    "AOT",
    "CM",
    "PIT",
    "PE",
    "CRT_Correct",
    "CRT_Heuristic",
]

for i in range(len(ind_variables)):
    var_name = ind_variables[i]
    print(f"VIF factor for {var_name}: {round(variance_inflation_factor(short_df[ind_variables], i), 2)}.")

VIF factor for AOT: 20.34.
VIF factor for CM: 7.52.
VIF factor for PIT: 10.57.
VIF factor for PE: 29.97.
VIF factor for CRT_Correct: 8.94.
VIF factor for CRT_Heuristic: 6.22.


In [13]:
# High multicollinearity. Run all models again and check differences between coefficients
subset_ind_variables = [
    "CM",
    "PIT",
    "CRT_Correct",
    "CRT_Heuristic",
]

for i in range(len(subset_ind_variables)):
    var_name = subset_ind_variables[i]
    print(f"VIF factor for {var_name}: {round(variance_inflation_factor(short_df[subset_ind_variables], i), 2)}.")

VIF factor for CM: 7.26.
VIF factor for PIT: 9.21.
VIF factor for CRT_Correct: 3.74.
VIF factor for CRT_Heuristic: 3.9.


In [14]:
subset_regressions = {}
predictors = ["AOT", "CM", "PIT", "PE", "CRT_Correct", "CRT_Heuristic"]
for i in ["MFQ_C", "MFQ_E", "MFQ_PRO", "MFQ_L", "MFQ_A", "MFQ_P"]:
    partial_X = short_df[subset_ind_variables]
    partial_X = sm.add_constant(partial_X)
    partial_model = sm.OLS(short_df[i], partial_X, hasconst=True).fit()
    subset_regressions[i] = {"full_model": partial_model}
    subset_regressions[i]["bootstrap"] = bootstrap_regression(partial_model, partial_X, short_df[i])

In [15]:
# Exploring differences in param. estimates and SE estimates between the full regression model
#      and the model with collinear variables excluded

largest_se_difference = 0
dep_var_se = None
input_var_se = None

largest_coef_difference = 0
dep_var_coef = None
input_var_coef = None

for var, results in subset_regressions.items():
    model = results["full_model"]
    for i, j in model.bse.items():
        if i == "const":
            continue
        original_se = regressions[var]["full_model"].bse.at[i]
        difference = abs(abs(original_se) - abs(model.bse.at[i]))
        if difference > largest_se_difference:
            largest_se_difference = difference
            dep_var_se = var
            input_var_se = i

        original_coef = regressions[var]["full_model"].params.at[i]
        difference = abs(abs(original_coef) - abs(model.params.at[i]))
        if difference > largest_coef_difference:
            largest_coef_difference = difference
            dep_var_coef = var
            input_var_coef = i
        

            
print(f"Largest SE difference was for {input_var_se} ({round(largest_se_difference, 4)}) for the model {dep_var_se}.")
print(f"Largest PARAM difference was for {input_var_coef} ({round(largest_coef_difference, 4)}) for the model {dep_var_coef}.")

Largest SE difference was for PIT (0.002) for the model MFQ_E.
Largest PARAM difference was for PIT (0.0614) for the model MFQ_P.
